## Temporal Patterns & Pandemic Analysis**Owner:** Jaismine Kaur &middot; **Date:** Oct 11, 2026

## Setup

In [ ]:
%matplotlib inlinefrom pathlib import Pathimport matplotlib.pyplot as pltimport pandas as pddf = pd.read_csv("../data/processed/crime_data_cleaned.csv")print(df.shape)df.head()

## 1. Time-of-Day PatternsEach crime type's hours are normalized to 100% so rare and common types canbe compared fairly.

In [ ]:
ct = pd.crosstab(df["TYPE"], df["HOUR"], normalize="index")print("Peak hour per crime type:")for t in ct.index:    peak = ct.loc[t].idxmax()    print(f"{t[:40]:40s} peak at {peak:02d}:00  ({ct.loc[t].max()*100:.1f}%)")

### Redacted TimestampsAlways check the raw hours of the violent categories before concluding.

In [ ]:
print(df[df["TYPE"].isin(["Homicide", "Offence Against a Person"])]["HOUR"].value_counts())

> **Finding 1 - Data forensics.** Every violent incident is stamped at> hour 00:00. VPD redacts the *time* of violent crimes as a privacy measure.> Hour-of-day analysis must exclude `Homicide` and `Offence Against a Person`.

### Night vs Daytime

In [ ]:
night = ct[[22, 23, 0, 1, 2, 3, 4, 5]].sum(axis=1) * 100day = ct[[9, 10, 11, 12, 13, 14, 15, 16, 17]].sum(axis=1) * 100print("Night (22-05) vs daytime (09-17) share per type:")for t in ct.index:    print(f"{t[:40]:40s} night {night[t]:4.1f}%   daytime {day[t]:4.1f}%")

### Hour Profiles

In [ ]:
for t in ct.index:    ct.loc[t].plot(figsize=(8, 2), title=t)    plt.xlabel("Hour"); plt.ylabel("Share of that type")    plt.tight_layout()    plt.show()

> **Finding 2 - Every crime has its own clock.** Commercial Break & Enter> is ~49% nocturnal (peak 04:00); Other Theft is ~58% daytime (peak 15:00);> collisions peak at 17:00 (rush hour). Crime type strongly predicts *when*> incidents happen.

## 2. COVID-19 Pandemic Effect2022 is missing from the dataset, so we compare 2019 (pre), 2020 (lockdown),2021, and 2023 (post).

In [ ]:
years = df[df["YEAR"].isin([2019, 2020, 2021, 2023])]y = pd.crosstab(years["TYPE"], years["YEAR"])print(y)

In [ ]:
# Pandemic shock: 2019 -> 2020change = (y[2020] - y[2019]) / y[2019] * 100print("Biggest changes 2019 -> 2020 (%):")print(change.sort_values().round(1))

In [ ]:
# Post-pandemic: 2021 -> 2023change2 = (y[2023] - y[2021]) / y[2021] * 100print("Biggest changes 2021 -> 2023 (%):")print(change2.sort_values().round(1))

In [ ]:
y.T[["Theft from Vehicle", "Other Theft", "Mischief",      "Break and Enter Commercial"]].plot(marker="o", figsize=(10, 5))plt.title("Pandemic and post-pandemic shift")plt.ylabel("Incidents"); plt.grid(alpha=0.3)plt.tight_layout()plt.show()

> **Finding 3 - The pandemic reshaped the mix.** In 2020, collisions with> injury fell -43% and vehicle-related thefts -38 to -39%, while Commercial> Break & Enter rose +12%. By 2023, Other Theft was +50% and Mischief +19%> versus 2021, while bicycle theft and residential break-ins kept falling.

## 3. SeasonalityHypothesis: Theft from Vehicle is worse in summer (more cars parked, windowsdown). Test it against the all-crime seasonal baseline.

In [ ]:
tfv = df[df["TYPE"] == "Theft from Vehicle"]season_order = ["Winter", "Spring", "Summer", "Fall"]s = tfv["SEASON"].value_counts().reindex(season_order)print("Theft from Vehicle by season:")for season in season_order:    print(f"  {season:8s} {s[season]:,}  ({s[season]/len(tfv)*100:.1f}%)")base = df["SEASON"].value_counts().reindex(season_order) / len(df) * 100print("\nAll crime baseline (%):")print(base.round(1))

> **Finding 4 - The hypothesis that failed.** Theft from Vehicle is flat> across seasons (24.2 / 24.7 / 25.3 / 25.7%) even though all crime has a> small summer bump (26.5% vs 23.6% in winter). The hypothesis is *rejected* -> still a result worth reporting.

## 4. Neighbourhood ChangeHow did incident counts change per neighbourhood between 2019 and 2023?

In [ ]:
hood = (df[df["YEAR"].isin([2019, 2023])]         .groupby(["NEIGHBOURHOOD", "YEAR"])         .size()         .unstack())hood["change_pct"] = (hood[2023] - hood[2019]) / hood[2019] * 100print("Top 5 improvers (% change 2019 -> 2023):")print(hood.nsmallest(5, "change_pct")[["change_pct"]].round(1))print("\nNeighbourhoods with increases:")print(hood[hood["change_pct"] > 0][["change_pct"]].round(1))

> **Finding 5 - Neighbourhoods diverge.** 21 of 24 neighbourhoods had> *fewer* incidents in 2023 than 2019. Biggest improvers: Oakridge (-50.7%),> Riley Park (-46.2%), Hastings-Sunrise (-42.7%). Only Musqueam (+25.8%),> Renfrew-Collingwood (+21.3%), and Sunset (+10.8%) increased.>> Caveats: Musqueam is small, so percentages are noisy; 2022 is missing;> population shifts can explain part of these changes.

### 5. Summary- Violent-crime timestamps are redacted (hour = 0) - exclude from time analysis- Crime type predicts time-of-day patterns - chi-square tests next- COVID changed the crime mix; the recovery is uneven- Theft from Vehicle has no seasonality (rejected hypothesis)- 21/24 neighbourhoods improved since 2019; 3 worsenedNext: move these findings into `app/pages/4_statistical.py` with properstatistical tests (`scipy`), and normalize neighbourhood comparisons bypopulation in the Crime Map phase.